# House Prices: EDA, Cleaning and Preprocessing

This notebook follows the attached workflow: explore, identify quality issues, clean, and prepare model-ready features. Run cells top to bottom from the project root.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
from src.process import clean
df_raw = pd.read_csv(ROOT / "data" / "house_prices.csv", low_memory=False)
print("Shape:", df_raw.shape)
display(df_raw.head())

## 1. Exploratory Data Analysis

Inspect types, descriptive statistics and missingness. Questions: Which cities have higher median asking prices? How skewed are prices? Which fields are unreliable?

In [ ]:
display(df_raw.dtypes.to_frame("dtype"))
display(df_raw.describe(include="all").T)
missing = df_raw.isna().mean().sort_values(ascending=False)
display((missing*100).round(1).to_frame("missing_percent"))
missing.head(15).sort_values().mul(100).plot.barh(figsize=(9,6),title="Missing percentage by field")
plt.show()

## 2. Cleaning

Rather than erasing possible duplicate listings, flag them. Convert price and area measurements to consistent numeric units. Drop only columns that are entirely empty or administrative.

In [ ]:
df = clean(df_raw)
print("Potential duplicate rows:",int(df.possible_duplicate_listing.sum()))
print("Missing prices:",int(df.price_inr.isna().sum()))
display(df[["Amount(in rupees)","price_inr","Carpet Area","carpet_area_sqft","bedrooms","bathroom_count"]].head())

## 3. Price distribution, location patterns and numeric correlations

In [ ]:
valid = df.loc[df.price_inr.gt(0)].copy()
fig, ax = plt.subplots(figsize=(9,4))
np.log10(valid.price_inr).hist(bins=60,ax=ax)
ax.set(title="House prices (log10 INR)",xlabel="log10(INR)",ylabel="Listings")
plt.show()
city = valid.groupby("location").price_inr.agg(["median","count"]).query("count >= 250").sort_values("count",ascending=False).head(12)
(city["median"]/1e5).sort_values().plot.barh(figsize=(9,5),title="Median price by city (lakh INR)")
plt.show()
correlation_cols=["price_inr","carpet_area_sqft","super_area_sqft","bathroom_count","balcony_count","bedrooms","floor_number","total_floors"]
display(valid[correlation_cols].corr())

## 4. Train-only imputation, scaling, encoding

Fit the preprocessing pipeline only on the training subset to avoid data leakage. `price_per_sqft_inr` is excluded, since it encodes pricing information.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
cat=["location","Furnishing","Transaction","Ownership","facing"]
num=["carpet_area_sqft","super_area_sqft","bathroom_count","balcony_count","bedrooms","floor_number","total_floors","parking_count"]
X=valid[cat+num].copy()
for c in cat: X[c]=X[c].astype("object").where(X[c].notna(),np.nan)
y=valid["price_inr"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
prep=ColumnTransformer([("num",Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("scaler",StandardScaler())]),num),("cat",Pipeline([("imputer",SimpleImputer(strategy="constant",fill_value="Unknown")),("encoder",OneHotEncoder(handle_unknown="ignore",min_frequency=100))]),cat)])
train_ready=prep.fit_transform(X_train)
test_ready=prep.transform(X_test)
print("Training",train_ready.shape,"Test",test_ready.shape)
print("Target distribution, train median INR:",y_train.median())

## 5. Conclusions and caveats

- The data include nonnumeric prices and very high missingness in some area fields.
- Potential duplicate listings should not be automatically deleted without a trustworthy property identifier.
- Group-aware validation is recommended before trusting a predictive model.
- Preprocessing is not a trained price prediction model.

See `reports/EDA_REPORT.md` for the findings and generated graphics.